<img src="https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/docs/assets/0.-BC-dev-hub-LOGO-flicker.svg" alt="BrainChip Dev Hub" width="200"/>

# Visual Wake Words (VWW) — Akida 2 Training

<p align="right">
Run Time: ~1 hour with training included / ~2 minutes with training skipped
</p>

This notebook walks through the full Akida 2 pipeline for the Visual Wake Words (person / non-person) task using an **MobileNet-0.25 model** at 96×96 resolution: float training, quantization with **quantizeml** (an 8-bit variant and a 4-bit QAT variant), conversion to Akida with **cnn2snn**, and evaluation on the Akida software backend.

The focus is on the **Akida-specific** aspects of the pipeline. The full data
preprocessing and training code is available in the accompanying Python files —
it is standard tf_keras code and is not described further in this notebook.

By default, model training is run to ensure reproducibility. However, you can
cut the running time of the notebook to under a minute if desired by 
skipping the training runs and evaluatinging the pretrained float and quantized models instead: simply set the relevant `RUN_FLOAT_TRAINING` and `RUN_QAT_TRAINING`
variables in the first code cell to `False`.

In [2]:
# Colab-only setup. Local users: ignore this cell — it does nothing for you.
import sys, os

if 'google.colab' in sys.modules:
    if not os.path.exists('colab_setup.py'):
        !wget -q https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/akida2/model_zoo/vww/colab_setup.py
    import colab_setup; colab_setup.setup()

## Setup

The default dataset path is `./data/vw_coco2014_96`. See the [README](README.md)
for download instructions and symlink setup. Update `DATA_PATH` below if needed.

In [3]:
import os
import numpy as np
import tensorflow as tf
from tqdm import tqdm

DATA_PATH = './data/vw_coco2014_96'
MODELS_DIR = './models'
os.makedirs(MODELS_DIR, exist_ok=True)

RUN_FLOAT_TRAINING = True
RUN_QAT_TRAINING = True

SEED = 42

# Must be called before any TF ops to make GPU ops deterministic.
tf.config.experimental.enable_op_determinism()

## Dataset

`get_data` returns a training and a validation `tf.data.Dataset`. The full
preprocessing and augmentation code is in [vww_data.py](vww_data.py) —
standard tf_keras pipeline code, not described further here.

One point to note: the data are delivered in the uint8 range of the original images, such that they can be fed to both `tf_keras` and `akida` models without any changes in scaling. See the Model description below for more details on how the inputs are scaled to the [-1, 1] range for training.

In [4]:
from vww_data import get_data

BATCH_SIZE = 32
INPUT_SHAPE = (96, 96, 3)

train_ds, val_ds = get_data(DATA_PATH, INPUT_SHAPE, BATCH_SIZE, seed=SEED)

Found 98658 images belonging to 2 classes.
Found 10961 images belonging to 2 classes.


## Model

We use a standard MobileNet, downloaded direct from `tf_keras` as the base model for this task. Note only that the code to build it adds a Rescaling layer up front: the pretrained ImageNets weights as provided expect inputs in the [-1, 1] range, whereas it can be sensible to feed images to Akida in their original uint8 format (avoiding a conversion step in a deployed environment). By incorporating the Rescaling for tf_keras within the model rather than the preprocessing pipeline, we can use a single pipeline to feed both model versions (it also helps to avoid user-errors at conversion time: because the Rescaling step is included in the model graph it can be automatically folded into the Akida layer parameters). 

In [5]:
from vww_model import build_vww_model

model = build_vww_model(seed=SEED)
model.summary()

I0000 00:00:1788361029.596293 2997675 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22284 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:04:00.0, compute capability: 8.6


Model: "mobilenet_vww"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 96, 96, 3)]       0         
                                                                 
 rescaling (Rescaling)       (None, 96, 96, 3)         0         
                                                                 
 conv1 (Conv2D)              (None, 48, 48, 8)         216       
                                                                 
 conv1_bn (BatchNormalizati  (None, 48, 48, 8)         32        
 on)                                                             
                                                                 
 conv1_relu (ReLU)           (None, 48, 48, 8)         0         
                                                                 
 conv_dw_1 (DepthwiseConv2D  (None, 48, 48, 8)         72        
 )                                                   

## Float Training

The model is trained for 20 epochs using Adam with a step-decay learning rate
schedule. The full training code is in [vww_train.py](vww_train.py) —
standard tf_keras code, not described further here.

In [6]:
from vww_train import train_vww

if RUN_FLOAT_TRAINING:
    LEARNING_RATE = 1e-3
    EPOCHS = 20
    # Freshly set the dataset seed for reproducibility
    train_ds, val_ds = get_data(DATA_PATH, INPUT_SHAPE, BATCH_SIZE, seed=SEED)

    train_vww(model, train_ds, val_ds, EPOCHS, LEARNING_RATE, seed=SEED)

    float_model_path = os.path.join(MODELS_DIR, 'mobilenet_vww.h5')
    model.save(float_model_path, include_optimizer=False)
    print(f'Float model saved to {float_model_path}')
else:
    from tf_keras.models import load_model
    print('Training skipped. Loading an existing float model...')
    model = load_model(os.path.join('pretrained_models', 'mobilenet_vww.h5'))

Found 98658 images belonging to 2 classes.
Found 10961 images belonging to 2 classes.
Epoch 1/20


I0000 00:00:1788361033.405258 2997834 cuda_dnn.cc:529] Loaded cuDNN version 90300


3084/3084 [==============================] - 130s 42ms/step - loss: 0.5758 - accuracy: 0.7457 - val_loss: 0.4413 - val_accuracy: 0.7977
Epoch 2/20
3084/3084 [==============================] - 130s 42ms/step - loss: 0.4050 - accuracy: 0.8167 - val_loss: 0.3817 - val_accuracy: 0.8350
Epoch 3/20
3084/3084 [==============================] - 127s 41ms/step - loss: 0.3824 - accuracy: 0.8288 - val_loss: 0.4080 - val_accuracy: 0.8143
Epoch 4/20
3084/3084 [==============================] - 126s 41ms/step - loss: 0.3666 - accuracy: 0.8365 - val_loss: 0.4110 - val_accuracy: 0.8181
Epoch 5/20
3084/3084 [==============================] - 128s 41ms/step - loss: 0.3490 - accuracy: 0.8458 - val_loss: 0.3658 - val_accuracy: 0.8377
Epoch 6/20
3084/3084 [==============================] - 126s 41ms/step - loss: 0.3371 - accuracy: 0.8534 - val_loss: 0.4222 - val_accuracy: 0.8084
Epoch 7/20
3084/3084 [==============================] - 126s 41ms/step - loss: 0.3272 - accuracy: 0.8574 - val_loss: 0.3206 - val

/home/dmclelland/miniconda3/envs/brainchip_devhub_env/lib/python3.12/site-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


### Evaluate float model

In [7]:
model.compile(metrics=['accuracy'])
_, float_accuracy = model.evaluate(val_ds, verbose=0)
print(f'Float validation accuracy: {float_accuracy:.4f}')

Float validation accuracy: 0.8809


## Quantization (quantizeml)

Akida 2 quantizes with **quantizeml** (not `cnn2snn.quantize`). We produce two variants:

* **8-bit** (i8/w8/a8) — post-training quantization only; 8-bit PTQ is accurate enough that QAT is not needed.
* **4-bit** (i8/w4/a4) — quantization-aware training (QAT); 4-bit PTQ accuracy is poor, so we fine-tune. The input layer weights stay 8-bit in both variants.

Note that quantization using `quantizeml` requires samples for calibration. Ideally those should be representative samples for the task, drawn from the training split to avoid data leakage.

In [8]:
from quantizeml.models import quantize, QuantizationParams

from vww_data import get_samples

NUM_SAMPLES = 1024
samples = get_samples(DATA_PATH, INPUT_SHAPE, num_samples=NUM_SAMPLES)

# --- 8-bit variant (i8 / w8 / a8), PTQ only ---
qparams_8bit = QuantizationParams(input_weight_bits=8, weight_bits=8, activation_bits=8)
model_8bit = quantize(model, qparams=qparams_8bit, samples=samples, batch_size=128, epochs=2)

q8_path = os.path.join(MODELS_DIR, 'mobilenet_vww_i8_w8_a8.h5')
model_8bit.save(q8_path, include_optimizer=False)

model_8bit.compile(metrics=['accuracy'])
_, acc_8bit = model_8bit.evaluate(val_ds, verbose=0)
print(f'8-bit quantized validation accuracy: {acc_8bit:.4f}')

Found 98658 images belonging to 2 classes.
8/8 [==============================] - 0s 3ms/step
8-bit quantized validation accuracy: 0.8791


Note: the quantized model can be saved using the standard method. However,
for later reloading, because of the custom quantized layers in the model
we have to use the `load_model` function from `quantizeml.model_io` (a wrapper
around the standard tf_keras loading function)

In [9]:
# Just to illustrate how to load a quantized model
del model_8bit

from quantizeml.model_io import load_model
model_8bit = load_model(q8_path)

### 4-bit variant with QAT

While 8-bit quantization usually results in a negligible loss of accuracy, that is rarely the case when quantizing aggressively to 4-bits. However, a few epochs of fine-tuning are typically sufficient to recover most of the accuracy lost. It is typical to find that a lower learning rate (e.g. /10) is required during this phase than during the initial training.

Note that, although Quatization Aware Training can sound intimidating,
the model quantized via `quantizeml` can simply be reinserted into the 
same training function that was used for the initial float training.

In [10]:
if RUN_QAT_TRAINING:
    # --- 4-bit variant (i8 / w4 / a4), QAT ---
    qparams_4bit = QuantizationParams(input_weight_bits=8, weight_bits=4, activation_bits=4)
    model_4bit = quantize(model, qparams=qparams_4bit, samples=samples, batch_size=128, epochs=2)

    # QAT fine-tune the quantized 4-bit model. quantizeml-quantized models are standard
    # Keras models, so the same training loop applies.
    QAT_EPOCHS = 5
    QAT_LR = 1e-4
    train_ds, val_ds = get_data(DATA_PATH, INPUT_SHAPE, BATCH_SIZE, seed=SEED)
    train_vww(model_4bit, train_ds, val_ds, QAT_EPOCHS, QAT_LR, seed=SEED)

    q4_path = os.path.join(MODELS_DIR, 'mobilenet_vww_i8_w4_a4_qat.h5')
    model_4bit.save(q4_path, include_optimizer=False)
else:
    # Note the use of the quantizeml load_model wrapper to handle the
    # custom quantized layer types
    from quantizeml.model_io import load_model
    print('Training skipped. Loading an existing float model...')
    model_4bit = load_model(os.path.join('pretrained_models', 'mobilenet_vww_i8_w4_a4_qat.h5'))

model_4bit.compile(metrics=['accuracy'])
_, acc_4bit = model_4bit.evaluate(val_ds, verbose=0)
print(f'4-bit QAT validation accuracy: {acc_4bit:.4f}')

8/8 [==============================] - 0s 3ms/step
Found 98658 images belonging to 2 classes.
Found 10961 images belonging to 2 classes.
Epoch 1/5
3084/3084 [==============================] - 200s 59ms/step - loss: 0.5165 - accuracy: 0.7527 - val_loss: 0.4200 - val_accuracy: 0.8011
Epoch 2/5
3084/3084 [==============================] - 174s 56ms/step - loss: 0.4062 - accuracy: 0.8135 - val_loss: 0.4375 - val_accuracy: 0.8007
Epoch 3/5
3084/3084 [==============================] - 179s 58ms/step - loss: 0.3806 - accuracy: 0.8282 - val_loss: 0.3805 - val_accuracy: 0.8252
Epoch 4/5
3084/3084 [==============================] - 176s 57ms/step - loss: 0.3640 - accuracy: 0.8362 - val_loss: 0.3608 - val_accuracy: 0.8357
Epoch 5/5
3084/3084 [==============================] - 178s 58ms/step - loss: 0.3519 - accuracy: 0.8417 - val_loss: 0.3550 - val_accuracy: 0.8428
4-bit QAT validation accuracy: 0.8428


## Conversion to Akida Format

`cnn2snn.convert` compiles the quantized Keras model into an Akida `.fbz`
model that can be loaded and executed directly on an Akida 2 hardware device.
The converter verifies hardware compatibility and maps each layer to its
corresponding Akida primitive. We convert both quantized variants.

In [11]:
from cnn2snn import convert

akida_8bit = convert(model_8bit)
akida_8bit.save(os.path.join(MODELS_DIR, 'mobilenet_vww_i8_w8_a8.fbz'))

akida_4bit = convert(model_4bit)
akida_4bit.save(os.path.join(MODELS_DIR, 'mobilenet_vww_i8_w4_a4_qat.fbz'))

akida_8bit.summary()

                Model Summary                 
______________________________________________
Input shape  Output shape  Sequences  Layers
[96, 96, 3]  [1, 1, 2]     1          30    
______________________________________________

______________________________________________________________
Layer (type)                  Output shape  Kernel shape    

========== SW/input_quantizer-dequantizer (Software) =========

input_quantizer (Quantizer)   [96, 96, 3]   N/A             
______________________________________________________________
conv1 (InputConv2D)           [48, 48, 8]   (3, 3, 3, 8)    
______________________________________________________________
conv_dw_1 (DepthwiseConv2D)   [48, 48, 8]   (3, 3, 8, 1)    
______________________________________________________________
conv_pw_1 (Conv2D)            [48, 48, 16]  (1, 1, 8, 16)   
______________________________________________________________
conv_dw_2 (DepthwiseConv2D)   [24, 24, 16]  (3, 3, 16, 1)   
______________________

## Evaluation on Akida (software backend)

We now run evaluation through the Akida model, to check that accuracy is 
comparable to that obtained from the quantized tf_keras model. Here, we deliberately use the software backend (the default, since we do not check 
for and map to a connected hardware device): this delivers a bit-accurate 
simulation of the  results that will be obtained when running the model on
hardware.

### Run Evaluation on Akida

The Akida runtime cannot consume `tf.data.Dataset` objects directly, rather
it expects a 4D numpy array (n, h, w, c). So we iterate over validation 
batches manually.

The model output tensor has shape `(B, 1, 1, C)` which is squeezed to 
`(B, C)` before taking the class argmax.

In [12]:
def evaluate_akida(akida_model, ds):
    ds.reset()
    labels_all, logits_all = [], []
    for _ in tqdm(range(len(ds)), desc='Evaluating on Akida'):
        batch, label_batch = next(ds)
        if not isinstance(batch, np.ndarray):
            batch = batch.numpy()
        logits = akida_model.predict(batch.astype(np.uint8)).squeeze(axis=(1, 2))
        labels_all.append(label_batch)
        logits_all.append(logits)
    labels_all = np.concatenate(labels_all)
    preds = np.argmax(np.concatenate(logits_all), axis=1)
    return float(np.mean(preds == labels_all))

akida_acc_8bit = evaluate_akida(akida_8bit, val_ds)
akida_acc_4bit = evaluate_akida(akida_4bit, val_ds)
print(f'Akida 8-bit accuracy: {akida_acc_8bit:.4f}')
print(f'Akida 4-bit QAT accuracy: {akida_acc_4bit:.4f}')

Evaluating on Akida: 100%|██████████| 343/343 [00:09<00:00, 35.15it/s]

Akida 8-bit accuracy: 0.8791
Akida 4-bit QAT accuracy: 0.8428


## Activation Sparsity

Activation sparsity drives efficiency on Akida (zero activations are skipped). To check activation sparsity within the model, we need to run some real samples through. Here, we re-use the samples that we generated earlier for calibration during quantization.

In [13]:
from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity

print('8-bit sparsity:')
pretty_print_sparsity(compute_sparsity(akida_8bit, samples=samples))
print('\n4-bit QAT sparsity:')
pretty_print_sparsity(compute_sparsity(akida_4bit, samples=samples))

8-bit sparsity:

Layer           Sparsity
------------------------
conv1            29.93%
conv_dw_1        37.78%
conv_pw_1        29.25%
conv_dw_2        14.57%
conv_pw_2        17.76%
conv_dw_3        28.46%
conv_pw_3        30.16%
conv_dw_4         6.00%
conv_pw_4         6.99%
conv_dw_5        36.02%
conv_pw_5        29.03%
conv_dw_6        13.09%
conv_pw_6        10.74%
conv_dw_7        38.44%
conv_pw_7        17.69%
conv_dw_8        32.85%
conv_pw_8        25.65%
conv_dw_9        30.64%
conv_pw_9        24.29%
conv_dw_10       35.28%
conv_pw_10       42.05%
conv_dw_11       36.88%
conv_pw_11       39.32%
conv_dw_12       22.46%
conv_pw_12       47.88%
conv_dw_13       37.72%
conv_pw_13       60.98%
predictions       0.00%
------------------------
Mean             27.93%

4-bit QAT sparsity:

Layer           Sparsity
------------------------
conv1            31.05%
conv_dw_1        39.50%
conv_pw_1        38.79%
conv_dw_2        17.22%
conv_pw_2        22.60%
conv_dw_3        34.

## Summary

In [14]:
print(f'{"Variant":<16}{"Keras acc":<12}{"Akida acc":<12}')
print(f'{"float":<16}{float_accuracy:<12.4f}{"-":<12}')
print(f'{"8-bit (w8a8)":<16}{acc_8bit:<12.4f}{akida_acc_8bit:<12.4f}')
print(f'{"4-bit QAT":<16}{acc_4bit:<12.4f}{akida_acc_4bit:<12.4f}')

Variant         Keras acc   Akida acc   
float           0.8809      -           
8-bit (w8a8)    0.8791      0.8791      
4-bit QAT       0.8428      0.8428      
